# BTS Digital Twin (NVS) — Vòng 1, chạy trên Kaggle

Notebook này KHÔNG nhúng code trực tiếp — nó `git clone` code từ repo GitHub của
bạn (thư mục `pipeline/`) và tải dataset từ Google Drive, rồi chạy. Ưu điểm: sửa
code xong chỉ cần `git push`, không cần regenerate lại file `.ipynb` này.

Tài liệu tham chiếu đầy đủ (nằm trong repo): `KE_HOACH_VONG1.md`, `Dataset/README.md`,
`pipeline/README.md`.

**Trước khi Run All, cần điền 2 chỗ:**
1. Settings (bên phải màn hình Kaggle) → Accelerator: chọn **GPU T4 x2** (hoặc P100) → Internet: **On**.
2. `REPO_URL` ở Bước 3 — link git repo chứa thư mục `pipeline/` (bạn sắp push lên).
3. `GDRIVE_URL` ở Bước 4 — đã điền sẵn link dataset bạn đưa.
4. Kaggle giới hạn ~30 giờ GPU/tuần, mỗi phiên chạy tối đa ~12 giờ — nên chạy từng
   scene một lần đầu, đừng Run All toàn bộ 13 scene ngay khi chưa chắc pipeline ổn
   (xem Bước 5 — Phase 0 bắt buộc trước tiên).

**Bảo mật:** để notebook Kaggle này ở chế độ **Private** (không Public/không share
link công khai) — nó chứa link Google Drive dataset của bạn. Token GitHub (nếu
repo code để Private) nên đi qua Kaggle Secrets như hướng dẫn ở Bước 3, không dán
thẳng vào code.


## Bước 1 — Cài đặt

In [ ]:
import torch, subprocess, sys
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG có GPU — vào Settings bật Accelerator GPU trước khi chạy tiếp")
print("Torch:", torch.__version__, "| CUDA build:", torch.version.cuda)


In [ ]:
!pip install -q pycolmap "scikit-image>=0.19" lpips plyfile tqdm gdown


## Bước 2 — Clone + build 3D Gaussian Splatting

Repo gốc `graphdeco-inria/gaussian-splatting` — dùng để train/render, không tự viết lại
trainer (quá nhiều chi tiết dễ sai: densification, SH coefficients...). Bước build
2 CUDA extension (`diff-gaussian-rasterization`, `simple-knn`) mất khoảng 2-5 phút.


In [ ]:
%cd /kaggle/working
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
!pip install -q ./gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q ./gaussian-splatting/submodules/simple-knn

import os
os.environ["GS_REPO"] = "/kaggle/working/gaussian-splatting"
print("GS_REPO =", os.environ["GS_REPO"])


## Bước 3 — Lấy code pipeline từ Git repo của bạn (khuyến nghị để **Private**)

Repo Private vẫn clone được bình thường trên Kaggle, chỉ cần xác thực bằng
**Personal Access Token (PAT)** thay vì mật khẩu. Các bước 1 lần:

1. Đẩy code lên GitHub, chọn **Private** khi tạo repo (không ai ngoài bạn xem được,
   kể cả khi bạn share notebook Kaggle này cho người khác sau này).
2. Tạo token: GitHub → **Settings → Developer settings → Personal access tokens →
   Fine-grained tokens → Generate new token**. Chọn:
   - Repository access: **Only select repositories** → chọn đúng repo vừa tạo.
   - Permissions → Contents: **Read-only** (không cần quyền gì khác).
   - Đặt ngày hết hạn (Expiration) ngắn thôi, vd 30-90 ngày — hết hạn thì tạo token mới.
3. **Copy token, dán vào Kaggle Secrets (KHÔNG dán thẳng vào code)**: trong notebook
   Kaggle, vào menu **Add-ons → Secrets → Add a new secret** → Label đặt đúng tên
   `GITHUB_TOKEN`, Value dán token vừa copy → Save. Cell bên dưới sẽ tự đọc secret
   này lúc chạy, token không hề xuất hiện trong code/notebook — kể cả nếu lỡ share
   notebook cho người khác, họ cũng không nhìn thấy được token của bạn.

Nếu push CẢ project (gồm `Đề bài.md`, `KE_HOACH_VONG1.md`, `Dataset/`, `pipeline/`...)
làm 1 repo cũng được — cell dưới tự dò tìm thư mục con tên `pipeline` (chứa `common/`
và `scripts/`) ở bất kỳ độ sâu nào trong repo, không cần đúng ngay gốc repo.


In [ ]:
REPO_URL = "https://github.com/ThongLuc2k3/BTS-Digital-Twin.git"

# GITHUB_TOKEN: ưu tiên lấy từ Kaggle Secrets (an toàn, không lộ trong code).
# Chỉ cần dán thẳng vào biến bên dưới nếu bạn KHÔNG dùng Kaggle Secrets (kém an
# toàn hơn — token sẽ nằm lộ trong notebook, đừng share notebook cho ai nếu làm vậy).
GITHUB_TOKEN = ""

try:
    from kaggle_secrets import UserSecretsClient
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
        print("Đã lấy GITHUB_TOKEN từ Kaggle Secrets.")
except Exception:
    if not GITHUB_TOKEN:
        print("Không tìm thấy Kaggle Secret 'GITHUB_TOKEN' (bỏ qua nếu repo Public, "
              "hoặc bạn đã dán token thẳng vào biến GITHUB_TOKEN ở trên).")

assert REPO_URL, "Chưa điền REPO_URL — dán link git repo chứa thư mục pipeline/ vào biến này rồi chạy lại cell."

clone_url = REPO_URL
if GITHUB_TOKEN and "github.com" in REPO_URL:
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

!rm -rf /kaggle/working/_repo_clone
!git clone --depth 1 "{clone_url}" /kaggle/working/_repo_clone


In [ ]:
# Tự dò thư mục "pipeline" (chứa common/ và scripts/) ở bất kỳ đâu trong repo vừa
# clone, rồi symlink về /kaggle/working/pipeline — mọi cell sau đều gọi script từ đây.
import os
import shutil
from pathlib import Path

CLONE_ROOT = Path("/kaggle/working/_repo_clone")
found = None
for dirpath, dirnames, filenames in os.walk(CLONE_ROOT):
    p = Path(dirpath)
    if p.name == "pipeline" and "common" in dirnames and "scripts" in dirnames:
        found = p
        break
if found is None and (CLONE_ROOT / "common").exists() and (CLONE_ROOT / "scripts").exists():
    found = CLONE_ROOT  # trường hợp bạn push thẳng NỘI DUNG pipeline/ làm gốc repo

if found is None:
    raise SystemExit(
        "Không tìm thấy thư mục 'pipeline' (chứa common/ và scripts/) trong repo vừa clone.\n"
        f"Nội dung clone nằm ở {CLONE_ROOT} — kiểm tra lại đã push đúng thư mục pipeline/ lên git chưa."
    )

print("Tìm thấy code pipeline tại:", found)
target = Path("/kaggle/working/pipeline")
if target.is_symlink():
    target.unlink()
elif target.exists():
    shutil.rmtree(target)
os.symlink(found.resolve(), target)
print("Đã symlink -> /kaggle/working/pipeline ->", found.resolve())

Path("/kaggle/working/pipeline/work").mkdir(parents=True, exist_ok=True)


## Bước 4 — Tải dataset từ Google Drive

Điền link chia sẻ Google Drive (chế độ "Anyone with the link") vào `GDRIVE_URL` bên
dưới — file phải là **1 file .zip** chứa `Dataset/VAI_NVS_DATA/phase1/{public_set,private_set1}/...`
(zip nguyên thư mục `Dataset` như trong repo, hoặc chỉ riêng `VAI_NVS_DATA` cũng
được — cell dưới tự dò tìm thư mục `phase1` ở bất kỳ độ sâu nào trong zip).


In [ ]:
GDRIVE_URL = "https://drive.google.com/file/d/1VCrLnw3fqrVpGBf3bMl0f2bSfX_LErMp/view?usp=drive_link"

assert GDRIVE_URL, "Chưa điền GDRIVE_URL — dán link chia sẻ Google Drive (Anyone with the link) của file zip dataset vào biến này rồi chạy lại cell."

import os
os.makedirs("/kaggle/working/_dataset_raw", exist_ok=True)
!gdown --fuzzy "{GDRIVE_URL}" -O /kaggle/working/dataset.zip
!unzip -q -o /kaggle/working/dataset.zip -d /kaggle/working/_dataset_raw
print("Đã giải nén xong, đang dò tìm thư mục phase1 ...")


In [ ]:
# Tự dò thư mục "phase1" (chứa public_set/ hoặc private_set1/) ở bất kỳ đâu trong
# zip vừa giải nén, rồi symlink về đúng vị trí mà pipeline/common/scenes.py cần:
#   /kaggle/working/Dataset/VAI_NVS_DATA/phase1
import os
from pathlib import Path

RAW_ROOT = Path("/kaggle/working/_dataset_raw")
found = None
for dirpath, dirnames, filenames in os.walk(RAW_ROOT):
    p = Path(dirpath)
    if p.name == "phase1" and (("public_set" in dirnames) or ("private_set1" in dirnames)):
        found = p
        break

if found is None:
    raise SystemExit(
        "Không tìm thấy thư mục 'phase1' chứa public_set/private_set1 trong zip vừa giải nén.\n"
        f"Nội dung giải nén nằm ở {RAW_ROOT} — kiểm tra lại cấu trúc zip bạn đã upload lên Google Drive."
    )

print("Tìm thấy:", found)
target_parent = Path("/kaggle/working/Dataset/VAI_NVS_DATA")
target_parent.mkdir(parents=True, exist_ok=True)
target = target_parent / "phase1"
if target.exists() or target.is_symlink():
    target.unlink() if target.is_symlink() else None
if not target.exists():
    os.symlink(found.resolve(), target)
print("Đã symlink ->", target, "->", found.resolve())

# QUAN TRỌNG: code (git clone) và dataset (Google Drive) không nằm chung 1 thư mục
# gốc trên Kaggle như lúc chạy local, nên common/scenes.py KHÔNG thể tự suy ra
# đường dẫn dataset bằng "đi lên N cấp từ vị trí file code" — phải khai báo thẳng
# qua biến môi trường này (đọc bởi pipeline/common/scenes.py).
os.environ["BTS_DATASET_ROOT"] = str(target.resolve())
print("Đã set BTS_DATASET_ROOT =", os.environ["BTS_DATASET_ROOT"])


In [ ]:
# Kiểm tra lại: liệt kê đủ 13 scene + scene nào có sparse gốc hợp lệ (chỉ HCM0249)
# Nếu train_ok=False hết cho mọi scene, kiểm tra lại giá trị BTS_DATASET_ROOT ở cell
# trên có trỏ đúng chỗ chứa public_set/private_set1 hay không.
import sys
sys.path.insert(0, "/kaggle/working/pipeline")
from common.scenes import all_scenes, DATASET_ROOT

print("DATASET_ROOT =", DATASET_ROOT, "| tồn tại:", DATASET_ROOT.exists())
for s in all_scenes():
    ok_train = s.train_images_dir.exists()
    ok_csv = s.test_poses_csv.exists()
    n_train = len(list(s.train_images_dir.glob("*"))) if ok_train else 0
    print(f"{s.name:10s} {s.split:8s} train_ok={ok_train} n_train={n_train:4d} "
          f"csv_ok={ok_csv} has_valid_provided_sparse={s.has_valid_provided_sparse()}")


## Bước 5 — PHASE 0: kiểm định hệ toạ độ (BẮT BUỘC chạy trước, xem KE_HOACH_VONG1.md mục 2)

Đây là rủi ro kỹ thuật lớn nhất của cả vòng thi: pose trong `test_poses.csv` có nằm
CÙNG hệ toạ độ với model 3D ta tự dựng từ COLMAP hay không. Đọc kỹ phần "KẾT LUẬN"
ở cuối output trước khi chạy tiếp các bước sau.


In [ ]:
!python /kaggle/working/pipeline/scripts/02_validate_frame.py


## Bước 6 — Thử nghiệm đầy đủ trên 1 scene public (kiểm tra pipeline trước khi chạy hàng loạt)

Dùng `hcm0031` (scene nhỏ nhất, 200 ảnh train / 50 pose test) để vòng lặp đầu nhanh hơn.

**Lưu ý:** từ đây output console chỉ còn 1-2 dòng tóm tắt mỗi scene (không còn log
nội bộ dài dòng của COLMAP hay progress bar 30.000 iteration của lúc train) — chi
tiết đầy đủ được ghi vào file, xem bảng ở `pipeline/README.md` mục 2 (vd
`pipeline/work/<scene>/train.log`, `.../colmap/colmap.log`, `.../render.log`).
Muốn xem tiến độ lúc train đang chạy: mở 1 cell khác gõ
`!tail -n 30 /kaggle/working/pipeline/work/hcm0031/train.log`.


In [ ]:
TEST_SCENE = "hcm0031"
!python /kaggle/working/pipeline/scripts/01_run_colmap.py --scene {TEST_SCENE}


In [ ]:
import os
os.environ["ITERATIONS"] = "30000"  # mặc định của repo gốc; giảm xuống vd 7000 nếu chỉ muốn xem nhanh có chạy được không
!bash /kaggle/working/pipeline/scripts/03_train_3dgs.sh {TEST_SCENE}


In [ ]:
!python /kaggle/working/pipeline/scripts/04_render_test_poses.py --scene {TEST_SCENE}
!python /kaggle/working/pipeline/scripts/05_eval_metrics.py --scene {TEST_SCENE}


Nếu PSNR/SSIM ở trên hợp lý (không phải ảnh nhiễu loạn ngẫu nhiên, PSNR quá thấp
~dưới 10-12) thì pipeline ổn — đi tiếp Bước 7/8. Nếu tệ bất thường, quay lại Bước 5
(khả năng cao là vấn đề hệ toạ độ hoặc COLMAP đăng ký thiếu ảnh — xem cảnh báo in
ra ở Bước 6 cell đầu, mục "Tỉ lệ đăng ký ảnh thấp").


## Bước 7 (tuỳ chọn) — Chạy nốt 4 scene public còn lại để có đánh giá đầy đủ

Có thể bỏ qua bước này nếu Bước 6 đã cho kết quả tốt và muốn tiết kiệm giờ GPU,
đi thẳng xuống Bước 8 (private set — bài nộp chính thức).


In [ ]:
PUBLIC_SCENES = ["HCM0181", "HCM0193", "HCM0204", "hcm0034"]  # đã làm hcm0031 ở Bước 6

for s in PUBLIC_SCENES:
    !python /kaggle/working/pipeline/scripts/01_run_colmap.py --scene {s}

import os
os.environ["ITERATIONS"] = "30000"
!bash /kaggle/working/pipeline/scripts/03_train_3dgs.sh {" ".join(PUBLIC_SCENES)}

for s in PUBLIC_SCENES:
    !python /kaggle/working/pipeline/scripts/04_render_test_poses.py --scene {s}

!python /kaggle/working/pipeline/scripts/05_eval_metrics.py --all_public


## Bước 8 — Chạy toàn bộ 8 scene private_set1 (bài nộp chính thức)

Đây là bước tốn thời gian nhất (8 scene × COLMAP + train 30000 iterations). Cân
nhắc giảm `ITERATIONS` nếu sắp hết giờ GPU trong tuần, hoặc chia làm nhiều session
(script tự bỏ qua bước COLMAP/database.db nếu đã chạy rồi — xem `common/colmap_runner.py`).


In [ ]:
PRIVATE_SCENES = ["HCM0249", "HCM0254", "HCM0276", "HCM1439",
                   "HNI0131", "HNI0265", "HNI0366", "HNI0437"]

for s in PRIVATE_SCENES:
    !python /kaggle/working/pipeline/scripts/01_run_colmap.py --scene {s}


In [ ]:
import os
os.environ["ITERATIONS"] = "30000"
!bash /kaggle/working/pipeline/scripts/03_train_3dgs.sh {" ".join(PRIVATE_SCENES)}


In [ ]:
for s in PRIVATE_SCENES:
    !python /kaggle/working/pipeline/scripts/04_render_test_poses.py --scene {s}


## Bước 9 — Đóng gói + kiểm tra `submission_round1.zip`

Script tự kiểm tra đủ 8 scene / đủ ảnh / đúng kích thước trước khi nén, và verify
lại chính file zip vừa tạo (xem `KE_HOACH_VONG1.md` mục 7 — checklist trước khi nộp).


In [ ]:
!python /kaggle/working/pipeline/scripts/06_package_submission.py \
    --out /kaggle/working/submission_round1.zip \
    --filename_mode literal


## Bước 10 — Lưu kết quả

- `submission_round1.zip` đã nằm ở `/kaggle/working/` — bấm **Save Version** (góc
  trên phải) để Kaggle giữ lại file này trong tab "Output" của notebook, tải về từ đó.
- Nếu muốn debug thêm sau này (xem lại sparse COLMAP, model đã train...), nén cả
  `pipeline/work/` lại và Save Version kèm theo — nhưng để ý dung lượng output của
  Kaggle có giới hạn (~20GB), không cần giữ nếu không có nhu cầu debug lại.


In [ ]:
import shutil
# (Tuỳ chọn) nén pipeline/work để tải về debug — CHỈ chạy nếu cần, có thể khá nặng.
# shutil.make_archive("/kaggle/working/pipeline_work_backup", "zip", "/kaggle/working/pipeline/work")

import os
print("Các file sẽ được giữ lại khi Save Version (trong /kaggle/working):")
for f in os.listdir("/kaggle/working"):
    print(" -", f)
